# 📗 합성곱 신경망: 손글씨 숫자를 맞히는 CNN 만들기

CNN(합성곱 신경망, convolutional neural network)은 3 × 3 같은 작은 필터로 그림을 한 칸씩 훑으며 이웃 픽셀을 묶어 선·모서리 같은 특징을 찾는 신경망입니다. 이 교안은 합성곱과 풀링을 작은 숫자 그림으로 직접 계산해 본 뒤, 둘을 쌓은 CNN으로 손글씨 숫자를 분류하고 학습한 모델을 저장합니다. 지난 시간 DNN과 같은 데이터와 설정으로 학습해 결과를 비교하고, 과적합 여부는 교안 02에서 검증 데이터로 가립니다.

<img src="images/60_lesson01_goals.png" width="1050" alt="제목은 '이번 교안에서 배울 것: 손글씨 숫자를 맞히는 CNN 만들기'이고, 제목 아래 띠에 'MNIST 손글씨 숫자 학습용 10,000장, 테스트용 2,000장'이 적혀 있습니다. 판 네 개가 2행 2열로 놓여 왼쪽 위 1, 오른쪽 위 2, 왼쪽 아래 3, 오른쪽 아래 4 순서로 화살표로 이어집니다. 1 '합성곱'(칩 '3 x 3 필터를 곱해서 더하기') 판에는 '5 x 5' 격자의 왼쪽 위 3 x 3 구역에 빨간 테두리가 있고 옆에 '3 x 3 필터'가 있습니다. 2 '풀링과 CNN 쌓기'(칩 'padding, 채널, 최댓값 풀링') 판에는 점점 작아지는 상자 다섯 개가 '(64, 1, 28, 28)'에서 '(64, 10)'으로 이어집니다. 3 '학습과 결과 읽기'(칩 '필터, 특징 맵, 틀린 그림') 판에는 흑백 '필터', 손글씨 7의 '특징 맵', 빨간 X가 붙은 '틀린 그림'이 있습니다. 4 '저장과 불러오기'(칩 '같은 예측 확인') 판에는 '모델'에서 'state_dict' 문서를 거쳐 '새 모델'로 가는 화살표가 있고, 개발자(초록 옷)가 새 모델을 가리킵니다.">


**학습 목표** (그림의 번호와 같습니다)

1. **그림을 (그림 수, 채널, 높이, 너비) 텐서로 바꾸고, 3 × 3 필터의 합성곱을 손으로 계산해 `nn.Conv2d` 결과와 맞춥니다.**
2. **padding·채널·최댓값 풀링이 출력 모양을 어떻게 바꾸는지 계산하고, 합성곱과 풀링을 쌓은 CNN을 `nn.Sequential`로 만듭니다.**
3. **지난 시간의 학습 함수로 CNN을 학습하고, 학습한 필터·특징 맵·틀린 그림으로 결과를 읽습니다.**
4. **`state_dict`로 학습한 모델을 저장하고 불러와 같은 예측을 내는지 확인합니다.**

## 1. 실습 준비

지난 시간의 DNN은 그림을 784개 한 줄로 펼쳐, 바로 위아래로 붙은 0행 5열과 1행 5열 픽셀이 5번과 33번 자리로 28칸 떨어졌습니다. CNN은 이웃 관계를 지키려고 그림을 가로·세로 모양 그대로 받으므로, 데이터를 불러와 CNN이 받는 모양으로 바꿉니다.

### 1-1. 라이브러리와 데이터

준비 셀은 지난 시간과 같습니다(계산 장치와 그래프 글꼴).

데이터는 손글씨 숫자 [MNIST](https://web.archive.org/web/20231231235959/http://yann.lecun.com/exdb/mnist/)(LeCun, Cortes & Burges, CC BY-SA 3.0)에서 고른 학습용 10,000장과 테스트용 2,000장입니다. 지난 시간과 같은 표본입니다.

In [ ]:
import platform
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import torch

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

In [ ]:
# torch.load는 그림과 정답 텐서 4개, 종류 이름 목록(class_names)이 든 딕셔너리를 돌려줍니다
mnist = torch.load(data_dir / "본실습" / "mnist_small.pt")
for name in ["train_images", "train_labels", "test_images", "test_labels"]:
    print(name, mnist[name].shape, mnist[name].dtype)
print("class_names:", mnist["class_names"])

In [ ]:
# 학습용 앞 10장을 정답과 함께 그립니다
class_names = mnist["class_names"]
plt.figure(figsize=(14, 2.2))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(mnist["train_images"][i], cmap="gray_r")  # gray_r: 0은 흰색, 255는 검은색
    plt.title(f"정답 {class_names[mnist['train_labels'][i]]}", fontsize=12)
    plt.axis("off")
plt.show()

### 1-2. CNN 입력 모양으로 바꾸기

`nn.Conv2d`는 그림 묶음을 4차원 텐서 (그림 수, 채널, 높이, 너비)로 받으므로, 28 × 28 그림 앞에 채널 축을 하나 끼웁니다.

- **채널(channel):** 픽셀 한 칸에 담긴 값의 개수입니다. 흑백 그림은 밝기 하나라 1, 컬러 사진은 빨강·초록·파랑 세 값이라 3입니다.
- **`unsqueeze(1)`:** 1번 자리(그림 수 다음)에 크기 1인 축을 끼웁니다. 값은 그대로이고 모양만 바뀝니다.


<img src="images/61_image_tensor_shape.png" width="1050" alt="제목은 'CNN이 받는 그림 묶음의 모양: (그림 수, 채널, 높이, 너비)'입니다. 왼쪽에 손글씨 7이 그려진 28 x 28 흑백 격자 한 장이 있고 '흑백: 채널 1'이 붙어 있습니다. 그 옆에 빨강, 초록, 파랑 판 세 장이 겹친 컬러 사진이 있고 '컬러: 채널 3'이 붙어 있습니다. 오른쪽에는 흑백 격자 카드가 비스듬히 여러 장 쌓여 있고, 쌓인 방향에 '그림 수 10,000', 카드의 세로에 '높이 28', 가로에 '너비 28'이 적혀 있습니다. 아래 띠에 '(10,000, 28, 28) → unsqueeze(1) → (10,000, 1, 28, 28)'이 적혀 있습니다.">


1. **불러온 그림:** (10,000, 28, 28), 0~255 정수(`uint8`)입니다.
2. **255로 나누기:** 지난 시간처럼 0~1 실수(`float32`)로 바꿉니다. 0은 0.0, 255는 1.0입니다.
3. **채널 축 끼우기:** (10,000, 1, 28, 28)이 됩니다. 컬러 사진이었다면 (10,000, 3, 28, 28)입니다.

In [ ]:
# 0~1 실수로 바꾸고, 1번 자리에 채널 축(흑백이라 1)을 끼웁니다
X_train = (mnist["train_images"].float() / 255).unsqueeze(1)
X_test = (mnist["test_images"].float() / 255).unsqueeze(1)
y_train = mnist["train_labels"]
y_test = mnist["test_labels"]

# 바뀐 모양과 자료형, 픽셀 값 범위를 확인합니다
print("X_train:", X_train.shape, X_train.dtype, "| y_train:", y_train.shape, y_train.dtype)
print("X_test:", X_test.shape, "| y_test:", y_test.shape)
print("픽셀 최솟값:", X_train.min().item(), "| 최댓값:", X_train.max().item())

## 2. 합성곱

CNN의 첫 계산은 필터로 그림을 훑는 합성곱이므로, 5 × 5 작은 그림으로 계산을 한 칸씩 따라갑니다.

### 2-1. 필터 하나의 계산

- **합성곱(convolution):** 3 × 3 필터를 그림의 3 × 3 구역에 겹쳐, 같은 자리 숫자끼리 곱해 모두 더한 값 하나를 냅니다. 필터를 한 칸씩 옮기며 모든 구역에서 이 계산을 합니다.
- **필터(filter):** 3 × 3 숫자 9개의 묶음입니다. 커널(kernel)이라고도 부릅니다. CNN에서는 이 숫자가 학습으로 정하는 가중치입니다.
- **특징 맵(feature map):** 필터가 그림 전체를 훑어 만든 결과 그림입니다. 필터가 찾는 모양이 있는 자리에서 값이 큽니다.
- **stride:** 필터를 한 번에 옮기는 칸 수입니다. 이 절의 합성곱은 모두 한 칸씩 옮깁니다(stride 1).


<img src="images/62_convolution_steps.png" width="1050" alt="제목은 '합성곱: 필터를 겹쳐 곱해서 더하고, 한 칸씩 옮깁니다'입니다. 왼쪽에 5 x 5 격자가 있고 가운데 세로줄(2열) 다섯 칸만 1, 나머지는 0입니다. 격자 왼쪽 위 3 x 3 구역에 주황 테두리 1, 한 칸 오른쪽 구역에 초록 테두리 2가 겹쳐 있습니다. 가운데에 3 x 3 필터 '-1, 2, -1 / -1, 2, -1 / -1, 2, -1'이 있습니다. 오른쪽 3 x 3 결과 격자는 세 행 모두 '-3, 6, -3'이고, 첫 칸 -3은 주황, 둘째 칸 6은 초록으로 칠해져 있습니다. 아래 계산 띠 두 줄: '주황: 행마다 0 × (-1) + 0 × 2 + 1 × (-1) = -1, 세 행 더해 -3', '초록: 행마다 0 × (-1) + 1 × 2 + 0 × (-1) = 2, 세 행 더해 6'.">


가운데 세로줄만 칠해진 5 × 5 그림(칠해진 칸 1, 빈칸 0)에 세로선 필터를 씁니다. 필터의 가운데 열은 2, 양옆 열은 -1입니다.

1. **왼쪽 위 구역(0~2행, 0~2열):** 행마다 픽셀 [0, 0, 1]과 필터 [-1, 2, -1]을 곱해 더하면 -1이고, 세 행을 더하면 -3입니다.
2. **한 칸 오른쪽 구역(0~2행, 1~3열):** 행마다 [0, 1, 0]이라 2이고, 세 행을 더하면 6입니다. 세로줄이 필터 가운데 열에 딱 겹친 자리입니다.
3. **결과 크기:** 5 × 5 그림에서 3 × 3 필터가 놓일 자리는 가로 3곳, 세로 3곳이라 특징 맵은 3 × 3입니다.

In [ ]:
# 5 x 5 그림: 가운데 세로줄(2열)만 1입니다
image_v = torch.zeros(5, 5)
image_v[:, 2] = 1
display(image_v)

# 세로선 필터: 가운데 열은 2, 양옆 열은 -1
filter_v = torch.tensor([[-1.0, 2.0, -1.0],
                         [-1.0, 2.0, -1.0],
                         [-1.0, 2.0, -1.0]])
display(filter_v)

In [ ]:
def convolve(image, kernel):
    """3 x 3 필터를 한 칸씩 옮기며 곱해서 더한 특징 맵을 돌려줍니다."""
    # 필터가 놓일 자리 수: 5 x 5 그림이면 5 - 3 + 1 = 3
    size = image.shape[0] - 2
    feature_map = torch.zeros(size, size)
    for row in range(size):
        for col in range(size):
            patch = image[row:row + 3, col:col + 3]  # 필터와 겹치는 3 x 3 구역
            feature_map[row, col] = (patch * kernel).sum()  # 같은 자리끼리 곱해 모두 더합니다
    return feature_map


# 세로줄 그림에 세로선 필터를 씁니다
v_result = convolve(image_v, filter_v)
display(v_result)

**결과 읽기**
- **가운데 열 6:** 세로줄이 필터 가운데(2)에 겹친 자리라 값이 가장 큽니다.
- **양옆 열 -3:** 세로줄이 필터 가장자리(-1)에 걸린 자리라 음수입니다. 세 행이 모두 [-3, 6, -3]으로 같습니다.

### 2-2. 세로선 필터와 가로선 필터

필터의 숫자 배치가 찾는 모양을 정하므로, 세로선 필터를 눕힌 가로선 필터를 만들어 세로줄 그림과 가로줄 그림에 각각 씁니다.

- **가로선 필터:** 가운데 행은 2, 위아래 행은 -1입니다.
- **세로선 필터에 가로줄이 겹치면:** 한 행의 세 칸이 모두 1이라 -1 + 2 - 1 = 0입니다. 필터 방향과 다른 선에는 반응하지 않습니다.

| 그림 | 세로선 필터 | 가로선 필터 |
|---|---|---|
| 세로줄 그림 | 가운데 열 6 | 모두 0 |
| 가로줄 그림 | 모두 0 | 가운데 행 6 |

In [ ]:
# 가로선 필터(세로선 필터를 눕힌 모양)와 가운데 가로줄(2행)만 1인 그림
filter_h = torch.tensor([[-1.0, -1.0, -1.0],
                         [2.0, 2.0, 2.0],
                         [-1.0, -1.0, -1.0]])
image_h = torch.zeros(5, 5)
image_h[2, :] = 1

# 그림 2장 x 필터 2개, 네 가지 결과를 차례로 봅니다
images = {"세로줄 그림": image_v, "가로줄 그림": image_h}
filters = {"세로선 필터": filter_v, "가로선 필터": filter_h}
for image_name in images:
    for filter_name in filters:
        print(image_name, "+", filter_name)
        display(convolve(images[image_name], filters[filter_name]))

같은 두 필터를 진짜 손글씨 7(학습용 1번 그림)에 씁니다. 그림이 28 × 28이라 필터가 놓일 자리는 28 - 3 + 1 = 26곳, 특징 맵은 26 × 26입니다. 음수는 지난 시간의 ReLU로 0으로 바꿔 필터가 찾은 자리만 남깁니다.

In [ ]:
# 학습용 1번 그림(숫자 7)의 28 x 28 픽셀에 두 필터를 쓰고, ReLU로 음수를 0으로 바꿉니다
digit = X_train[1, 0]
v_digit_map = torch.relu(convolve(digit, filter_v))
h_digit_map = torch.relu(convolve(digit, filter_h))
print("원본:", tuple(digit.shape), "| 특징 맵:", tuple(v_digit_map.shape))

# 왼쪽부터 원본, 세로선 필터의 특징 맵, 가로선 필터의 특징 맵
plt.figure(figsize=(14, 4.8))
panels = {"원본 (정답 7)": digit, "세로선 필터의 특징 맵": v_digit_map, "가로선 필터의 특징 맵": h_digit_map}
for k, title in enumerate(panels):
    plt.subplot(1, 3, k + 1)
    plt.imshow(panels[title], cmap="gray_r")
    plt.title(title, fontsize=14)
    plt.axis("off")
plt.show()

**그래프 읽기**
- **세로선 필터:** 오른쪽 위에서 왼쪽 아래로 내려오는 긴 획이 진하게 남고, 맨 위 가로획은 양 끝만 조금 남습니다.
- **가로선 필터:** 맨 위 가로획이 진하게 남고, 긴 세로 방향 획은 흐려집니다.
- 같은 그림이라도 필터마다 다른 획을 골라냅니다. CNN은 이런 필터 숫자를 사람이 정하지 않고 학습으로 정합니다.

### 2-3. nn.Conv2d로 같은 계산 확인

PyTorch의 합성곱 층이 위에서 직접 한 계산과 같은지, 같은 필터 숫자를 넣어 확인합니다.

- **`nn.Conv2d(in_channels, out_channels, kernel_size)`:** 합성곱 층입니다. 입력 채널 수, 출력 채널 수(필터 수), 필터 크기를 받습니다. 필터 숫자(`weight`)와 절편(`bias`)이 파라미터입니다.
- **`weight` 모양:** (출력 채널, 입력 채널, 3, 3)입니다. 필터 1개가 흑백 1채널을 보면 (1, 1, 3, 3)입니다.
- 학습 대신 세로선 필터 숫자를 `weight`에 직접 넣고, 필터 계산만 비교하려고 절편은 뺍니다(`bias=False`).

In [ ]:
import torch.nn as nn

# 입력 채널 1개, 출력 채널 1개(필터 1개), 3 x 3 필터인 합성곱 층
conv = nn.Conv2d(1, 1, kernel_size=3, bias=False)
print("weight 모양:", conv.weight.shape)

# 세로선 필터 숫자를 (출력 채널, 입력 채널, 3, 3) 모양으로 넣습니다
with torch.no_grad():
    conv.weight[:] = filter_v.reshape(1, 1, 3, 3)

    # 5 x 5 그림도 (그림 수 1, 채널 1, 5, 5) 모양으로 넣습니다
    conv_output = conv(image_v.reshape(1, 1, 5, 5))
display(conv_output)
print("직접 계산과 같은가:", torch.equal(conv_output[0, 0], v_result))

**결과 읽기**
- **직접 계산과 같은가 True:** 손으로 짠 `convolve`와 `nn.Conv2d`가 같은 값을 냅니다. 합성곱 층은 위에서 계산한 일을 그대로 합니다.

### 2-4. padding과 출력 크기

필터가 놓일 자리만 계산하면 층을 지날 때마다 가장자리가 한 줄씩 깎이므로, 그림 둘레에 0을 둘러 크기를 지킵니다.

- **padding:** 그림 테두리에 0을 몇 겹 두르는지입니다. `padding=1`이면 한 겹이라 5 × 5 그림이 7 × 7이 되고, 가장자리 픽셀도 필터 가운데에 놓입니다.
- **출력 크기:** 입력 크기 $n$, 필터 크기 $f$, padding $p$일 때 출력 크기 $o$는, 필터가 놓일 자리 수 $n - f + 1$에 양쪽 padding $2p$칸을 더한 값입니다(stride 1).

$\displaystyle o = n - f + 1 + 2p$


<img src="images/63_padding.png" width="1050" alt="제목은 'padding: 테두리에 0을 둘러 크기를 지킵니다'입니다. 왼쪽 판 'padding=0'에는 '5 x 5' 격자의 왼쪽 위에 3 x 3 필터가 겹쳐 있고, 결과 '3 x 3' 격자와 '5 - 3 + 1 = 3'이 있습니다. 오른쪽 판 'padding=1'에는 5 x 5 둘레에 0 칸을 한 겹 둘러 '7 x 7'이 된 격자가 있습니다. 필터는 맨 왼쪽 위 모서리의 0 칸에 걸쳐 있고, 필터 가운데 칸인 원래 그림의 왼쪽 위 픽셀에 '가장자리 픽셀이 필터 가운데' 말풍선이 붙어 있습니다. 결과 '5 x 5' 격자와 '5 - 3 + 1 + 2 × 1 = 5'가 이어지고, 아래 띠에 '출력 크기 = 입력 - 필터 + 1 + 2 × padding'이 적혀 있습니다.">


1. **5 × 5, padding 0:** 5 - 3 + 1 = 3입니다. 앞에서 직접 계산한 특징 맵이 3 × 3이었던 까닭입니다.
2. **5 × 5, padding 1:** 5 - 3 + 1 + 2 = 5로 크기가 그대로입니다.
3. **28 × 28, padding 1:** 28 그대로입니다. 이 교안의 CNN은 합성곱에서 `padding=1`로 크기를 지키고, 크기를 줄이는 일은 다음 절의 풀링에 맡깁니다.

In [ ]:
# padding을 0, 1, 2로 바꾸며 5 x 5 그림과 28 x 28 그림의 출력 크기를 봅니다
size_rows = []
for padding in [0, 1, 2]:
    pad_conv = nn.Conv2d(1, 1, kernel_size=3, padding=padding)
    with torch.no_grad():
        small_output = pad_conv(torch.zeros(1, 1, 5, 5))
        digit_output = pad_conv(X_train[:1])
    size_rows.append({"padding": padding,
                      "5 x 5 입력의 출력": tuple(small_output.shape[2:]),
                      "28 x 28 입력의 출력": tuple(digit_output.shape[2:])})

# padding마다 출력 크기를 표로 봅니다
size_table = pd.DataFrame(size_rows)
display(size_table)

**표 읽기**
- **padding 1:** 5 × 5와 28 × 28 모두 크기가 그대로라, 합성곱 층을 여러 번 쌓아도 크기가 줄지 않습니다.
- **padding 2:** 7 × 7과 30 × 30으로 입력보다 커집니다. 식의 $2p$가 4가 되기 때문입니다.

### 2-5. 채널과 특징 맵

필터 하나는 한 가지 모양만 찾으므로, 실제 CNN은 한 층에 필터를 여러 개 두어 여러 특징을 함께 찾습니다.

- **출력 채널 수 = 필터 수 = 특징 맵 수:** `nn.Conv2d(1, 16, ...)`은 필터 16개가 각자 그림을 훑어 특징 맵 16장을 만듭니다. 그림 64장이면 (64, 16, 28, 28)입니다.
- **다음 층의 필터:** `nn.Conv2d(16, 32, ...)`의 필터 하나는 3 × 3 × 16 숫자입니다. 특징 맵 16장의 같은 3 × 3 자리를 한꺼번에 곱해 모두 더하므로, 앞 층이 찾은 여러 특징을 묶어 더 복잡한 모양을 찾습니다.


<img src="images/64_channels.png" width="1050" alt="제목은 '채널: 필터 수만큼 특징 맵이 생깁니다'입니다. 왼쪽에 손글씨 7 흑백 그림 한 장 '(1, 28, 28)'이 있습니다. 그 오른쪽에 색이 다른 작은 3 x 3 필터들이 세로로 늘어서 있고 'nn.Conv2d(1, 16): 필터 16개'가 적혀 있으며, 필터마다 화살표가 한 묶음으로 쌓인 특징 맵 '(16, 28, 28)'로 모입니다. 다음에 3 x 3 칸이 층층이 쌓인 기둥 모양 필터 '3 x 3 x 16'과 'nn.Conv2d(16, 32): 필터 하나가 16장을 한꺼번에 봅니다'가 있습니다. 맨 오른쪽에 더 두껍게 쌓인 특징 맵 묶음 '(32, 28, 28)'과 '필터 32개 -> 특징 맵 32장'이 있습니다.">


1. **첫째 층 `nn.Conv2d(1, 16)`:** 필터 16개 × 숫자 9개(1 × 3 × 3) = 가중치 144개, 절편 16개를 더해 파라미터 160개입니다.
2. **둘째 층 `nn.Conv2d(16, 32)`:** 필터 32개 × 숫자 144개(16 × 3 × 3) = 가중치 4,608개, 절편 32개를 더해 4,640개입니다.
3. **모양:** 그림 64장이 (64, 1, 28, 28) → (64, 16, 28, 28) → (64, 32, 28, 28)로 바뀝니다. 그림 수와 크기는 그대로이고 채널 수만 바뀝니다.

In [ ]:
# 첫째 층은 흑백 1채널을 받아 특징 맵 16장, 둘째 층은 16장을 받아 특징 맵 32장을 만듭니다
conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)

# 학습용 앞 64장을 두 층에 차례로 넣습니다
with torch.no_grad():
    maps1 = conv1(X_train[:64])
    maps2 = conv2(maps1)

# 층마다 weight 모양, 파라미터 수(weight + bias), 출력 모양을 표로 봅니다
channel_table = pd.DataFrame([
    {"층": "입력", "weight 모양": "-", "파라미터 수": 0, "출력 모양": tuple(X_train[:64].shape)},
    {"층": "nn.Conv2d(1, 16)", "weight 모양": tuple(conv1.weight.shape),
     "파라미터 수": conv1.weight.numel() + conv1.bias.numel(), "출력 모양": tuple(maps1.shape)},
    {"층": "nn.Conv2d(16, 32)", "weight 모양": tuple(conv2.weight.shape),
     "파라미터 수": conv2.weight.numel() + conv2.bias.numel(), "출력 모양": tuple(maps2.shape)},
])
display(channel_table)

**표 읽기**
- **`weight` 모양:** (16, 1, 3, 3)과 (32, 16, 3, 3)으로, 앞의 두 숫자가 출력 채널 수와 입력 채널 수입니다.
- **파라미터 수 160개, 4,640개와 출력 모양:** 위 계산과 같고, 그림 수 64와 크기 28 × 28은 그대로이며 채널 수만 16, 32로 바뀝니다.

## 3. 풀링

특징 맵이 32장으로 늘면 계산할 값도 그만큼 늘어나므로, 찾은 특징은 남기고 크기를 줄입니다.

### 3-1. 2 × 2 최댓값 풀링

- **풀링(pooling):** 특징 맵을 작은 구역으로 나눠, 구역마다 값 하나만 남겨 크기를 줄이는 계산입니다. 파라미터가 없습니다.
- **최댓값 풀링(max pooling):** 겹치지 않는 2 × 2 구역마다 가장 큰 값을 남깁니다. 구역 크기만큼 2칸씩 옮기므로 stride 2이고, 가로와 세로가 절반이 되어 값의 개수는 4분의 1이 됩니다.
- **최댓값을 남기는 이유:** 특징 맵의 큰 값은 필터가 찾는 모양이 있다는 뜻입니다. 그 모양이 구역 안에서 한 칸 옮겨져도 최댓값은 같아서, 획의 위치가 조금 달라도 같은 특징으로 봅니다.


<img src="images/65_max_pooling.png" width="1050" alt="제목은 '최댓값 풀링: 2 x 2 구역마다 가장 큰 값만 남깁니다'입니다. 왼쪽에 4 x 4 격자 '1, 3, 0, 2 / 4, 2, 1, 0 / 0, 1, 5, 6 / 2, 0, 7, 1'이 있고, 2 x 2 구역 네 개가 각각 파랑, 주황, 초록, 보라 배경으로 칠해져 있습니다. 구역마다 가장 큰 값 4, 2, 2, 7에 동그라미가 쳐져 있습니다. 오른쪽에 같은 네 색의 2 x 2 격자 '4, 2 / 2, 7'이 있고, 가운데 화살표 위에 'nn.MaxPool2d(2)'가 적혀 있습니다. 아래 띠에 '가로, 세로가 절반: 28 x 28 → 14 x 14 → 7 x 7'이 적혀 있습니다.">


1. **왼쪽 위 구역** [1, 3, 4, 2]의 최댓값은 4, **오른쪽 위** [0, 2, 1, 0]은 2입니다.
2. **왼쪽 아래** [0, 1, 2, 0]은 2, **오른쪽 아래** [5, 6, 7, 1]은 7입니다.
3. 4 × 4 특징 맵이 2 × 2 [[4, 2], [2, 7]]이 됩니다.

In [ ]:
# 4 x 4 특징 맵 예시
feature = torch.tensor([[1.0, 3.0, 0.0, 2.0],
                        [4.0, 2.0, 1.0, 0.0],
                        [0.0, 1.0, 5.0, 6.0],
                        [2.0, 0.0, 7.0, 1.0]])

# 겹치지 않는 2 x 2 구역마다 가장 큰 값을 남깁니다
pooled = torch.zeros(2, 2)
for row in range(2):
    for col in range(2):
        block = feature[row * 2:row * 2 + 2, col * 2:col * 2 + 2]  # 2칸씩 건너뛰며 구역을 고릅니다
        pooled[row, col] = block.max()
display(pooled)

In [ ]:
# nn.MaxPool2d(2): 2 x 2 구역의 최댓값 풀링 층. 입력은 (그림 수, 채널, 높이, 너비) 모양입니다
pool = nn.MaxPool2d(2)
pool_output = pool(feature.reshape(1, 1, 4, 4))
display(pool_output)
print("직접 계산과 같은가:", torch.equal(pool_output[0, 0], pooled))

### 3-2. 손글씨 특징 맵에 풀링하기

28 × 28 특징 맵에 풀링을 두 번 하면 14 × 14, 7 × 7이 됩니다. 숫자 7의 세로선 특징 맵(`padding=1`이라 28 × 28)으로 크기가 줄어도 획 모양이 남는지 봅니다.

In [ ]:
# 세로선 필터를 넣은 합성곱 층(padding=1)으로 숫자 7의 28 x 28 특징 맵을 만들고 ReLU를 씁니다
v_conv = nn.Conv2d(1, 1, kernel_size=3, padding=1, bias=False)
with torch.no_grad():
    v_conv.weight[:] = filter_v.reshape(1, 1, 3, 3)
    v_map = torch.relu(v_conv(X_train[1:2]))  # X_train[1:2]: 숫자 7 한 장, 모양 (1, 1, 28, 28)

    # 최댓값 풀링을 두 번 합니다
    pooled_once = pool(v_map)
    pooled_twice = pool(pooled_once)

# 왼쪽부터 풀링 전 28 x 28, 한 번 뒤 14 x 14, 두 번 뒤 7 x 7
plt.figure(figsize=(14, 4.8))
panels = {"풀링 전": v_map, "풀링 1번 뒤": pooled_once, "풀링 2번 뒤": pooled_twice}
for k, title in enumerate(panels):
    plt.subplot(1, 3, k + 1)
    plt.imshow(panels[title][0, 0], cmap="gray_r")
    plt.title(f"{title}: {tuple(panels[title].shape[2:])}", fontsize=14)
    plt.axis("off")
plt.show()

**그래프 읽기**
- **칸 수:** 784개에서 196개, 49개로 줄어듭니다.
- **획의 자리:** 7 × 7에서도 왼쪽 아래로 내려오는 긴 획의 자리가 진한 칸으로 남습니다. 뒤 층은 이 작은 특징 맵으로 어디쯤에 어떤 획이 있는지 계산합니다.

## 4. CNN 쌓기

합성곱으로 특징을 찾고 풀링으로 줄이는 묶음을 두 번 쌓은 뒤, 마지막에 지난 시간처럼 `nn.Linear`로 숫자 10개의 로짓을 냅니다.

### 4-1. nn.Sequential로 CNN 만들기

- **합성곱 묶음:** `nn.Conv2d` → `nn.ReLU` → `nn.MaxPool2d(2)`입니다. 합성곱도 곱해서 더하는 직선 계산이라, 지난 시간처럼 층 사이에 ReLU를 넣습니다.
- **`nn.Flatten()`:** 마지막 특징 맵 (32, 7, 7)을 숫자 1,568개(32 × 7 × 7) 한 줄로 펼칩니다. 합성곱 층이 이웃 픽셀로 특징을 다 찾은 뒤에 펼치므로, 처음부터 픽셀을 펼친 DNN과 다릅니다.
- **로짓(logit):** 출력층이 칸마다 내는 값입니다. softmax를 거치면 확률이 됩니다.
- **손실:** 지난 시간 다중 분류의 짝(로짓 10개 + `nn.CrossEntropyLoss`)을 그대로 씁니다.


<img src="images/66_cnn_structure.png" width="1050" alt="제목은 'CNN: 특징을 찾고 줄이기를 두 번, 마지막에 분류'입니다. 위쪽 '특징 찾기' 판에서 손글씨 7 그림 '(1, 28, 28)'이 파란 상자 '합성곱 묶음 1: Conv2d(1, 16) + ReLU + MaxPool2d(2)'를 지나 판 16장 '(16, 14, 14)'가 되고, '합성곱 묶음 2: Conv2d(16, 32) + ReLU + MaxPool2d(2)'를 지나 판 32장 '(32, 7, 7)'이 됩니다. 화살표가 아래쪽 '분류' 판으로 이어져 회색 상자 'Flatten: 1,568개 한 줄', 초록 상자 'Linear(1568, 10)'을 지나 '로짓 10개' 막대그래프(0~9, 7 막대만 주황으로 가장 김)가 됩니다.">

In [ ]:
def make_cnn():
    """합성곱 묶음 두 개(필터 16개, 32개)와 출력층으로 된 CNN을 만들어 계산 장치에 올려 돌려줍니다."""
    return nn.Sequential(
        nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # (16, 14, 14)
        nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # (32, 7, 7)
        nn.Flatten(),                                                             # 1,568개 한 줄
        nn.Linear(32 * 7 * 7, 10),                                                # 로짓 10개
    ).to(device)


# 파라미터 초깃값을 고정해 다시 실행해도 같은 모델이 나오게 합니다
torch.manual_seed(0)
model = make_cnn()
print(model)

In [ ]:
# 층마다 파라미터 모양과 개수. 이름의 앞 숫자는 nn.Sequential 안의 순서입니다
param_rows = []
for name, param in model.named_parameters():
    param_rows.append({"이름": name, "모양": tuple(param.shape), "개수": param.numel()})
param_table = pd.DataFrame(param_rows)
display(param_table)
print("파라미터 수 합계:", param_table["개수"].sum())

**결과 읽기**
- **합성곱 층(`0`, `3`):** 160개와 4,640개입니다. 필터 숫자를 그림의 모든 자리에서 함께 쓰므로, 그림이 커져도 개수가 늘지 않습니다.
- **출력층(`7.weight`):** (10, 1568), 15,680개로 가장 많습니다.
- **합계 20,490개:** 지난 시간 DNN(109,386개)의 약 5분의 1입니다.

### 4-2. 층마다 모양 변화

어느 층에서 모양이 어떻게 바뀌는지 알아야 `nn.Linear`의 입력 크기(32 × 7 × 7)를 맞출 수 있으므로, 미니배치 하나를 층마다 차례로 통과시켜 모양을 기록합니다. 미니배치를 꺼내는 `DataLoader`는 지난 시간과 같게 만듭니다(64장씩, 학습용만 섞기).

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# 학습용은 에폭마다 섞고, 테스트용은 결과만 계산하므로 섞지 않습니다
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=64, shuffle=False)
print("학습 미니배치 수:", len(train_loader), "| 테스트 미니배치 수:", len(test_loader))

In [ ]:
# 학습용 미니배치 하나(64장)를 꺼냅니다
for X_batch, y_batch in train_loader:
    break

# nn.Sequential의 층을 하나씩 꺼내 차례로 통과시키며 출력 모양을 기록합니다
shape_rows = [{"번호": "-", "층": "입력", "출력 모양": tuple(X_batch.shape)}]
x = X_batch.to(device)
with torch.no_grad():
    for k, layer in enumerate(model):
        x = layer(x)
        shape_rows.append({"번호": k, "층": type(layer).__name__, "출력 모양": tuple(x.shape)})  # 층 종류 이름만 적습니다

# 층마다 출력 모양을 표로 봅니다
shape_table = pd.DataFrame(shape_rows)
display(shape_table)

**결과 읽기**
- **번호:** `print(model)`의 (0)~(7)과 같습니다.
- **합성곱(`Conv2d`, 0번과 3번):** 채널 수만 1 → 16, 16 → 32로 바뀌고 크기(28, 14)는 그대로입니다(`padding=1`).
- **풀링(`MaxPool2d`, 2번과 5번):** 채널 수는 그대로이고 크기만 28 → 14, 14 → 7로 절반이 됩니다(stride 2).
- **`Flatten`(6번)과 `Linear`(7번):** (64, 32, 7, 7)이 (64, 1568)로 펼쳐지고, 마지막에 그림 64장마다 로짓 10개인 (64, 10)이 됩니다. `nn.Linear`의 입력 크기를 1,568이 아닌 값으로 적으면 이 줄에서 모양이 맞지 않는다는 에러가 납니다.

### 4-3. 첫 손실 점검

지난 시간처럼 학습 전 첫 손실이 $\log 10 \approx 2.30$ 근처인지 먼저 봅니다.

In [ ]:
import math

# 지난 시간 다중 분류와 같은 교차 엔트로피 손실. 교차 엔트로피 = 평균(-log(정답 칸의 확률))
loss_fn = nn.CrossEntropyLoss()

# 학습용 10,000장 전체의 첫 손실. 점검만 하므로 기울기 기록은 끕니다
with torch.no_grad():
    first_loss = loss_fn(model(X_train.to(device)), y_train.to(device)).item()
print("첫 손실:", round(first_loss, 4), "| log(10):", round(math.log(10), 4))

**결과 읽기**
- **첫 손실 2.3075:** $\log 10 = 2.3026$과 거의 같습니다. 출력층 뉴런 수(10)와 손실의 연결이 맞으므로 학습을 시작합니다.

## 5. 학습하기

모델과 손실이 준비됐으므로, 지난 시간의 `train_one_epoch`로 CNN을 학습합니다.

### 5-1. 10에폭 학습하기

- **설정:** Adam 학습률 0.001, 배치 크기 64, 10에폭입니다. 지난 시간 DNN과 같은 설정이고 모델만 CNN으로 바꿨습니다.
- **기록:** 에폭마다 학습 손실과 학습 데이터 정확도를 기록합니다. 학습 데이터 정확도는 학습용 10,000장을 한 번에 모델에 넣어 argmax 예측으로 계산합니다(지난 시간과 같습니다). 테스트 데이터는 학습을 마친 뒤 한 번만 씁니다.
- **실행 환경:** 계산 장치와 스레드 수에 따라 소수 셋째 자리, 틀린 그림의 수와 번호, 필터 번호가 달라질 수 있습니다. 결론은 곡선의 모양과 큰 차이로 읽습니다.
- **학습 전 기록:** 학습 전·후를 비교하려고, 학습을 시작하기 전에 첫 층 필터와 테스트 그림 10장의 예측을 저장해 둡니다.
- CPU에서 에폭마다 몇 초씩 걸립니다.

학습 함수와 평가 함수는 지난 시간에 만든 것을 그대로 씁니다.

In [ ]:
# 지난 시간에 만든 학습 함수와 평가 함수를 그대로 씁니다
def train_one_epoch(model, loader, loss_fn, optimizer, scheduler=None):
    """미니배치마다 파라미터를 한 번씩 갱신하고, 그 에폭 미니배치 손실의 평균을 돌려줍니다."""
    total_loss = 0.0
    for X_batch, y_batch in loader:
        # 미니배치를 모델이 있는 계산 장치로 옮깁니다
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # 지난 스텝의 기울기를 지우고, 순전파 -> 손실 -> 역전파 -> 갱신
        optimizer.zero_grad()
        loss = loss_fn(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()

        # 스케줄러가 있으면 스텝마다 학습률을 바꿉니다
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()

    # len(loader)는 미니배치 수입니다
    return total_loss / len(loader)


def evaluate(model, loader, loss_fn):
    """기울기 없이 예측해 (평균 손실, 정확도)를 돌려줍니다."""
    total_loss = 0.0
    correct = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            logits = model(X_batch)

            # 마지막 미니배치는 장 수가 적으므로, 손실에 장 수를 곱해 더한 뒤 전체 장 수로 나눕니다
            total_loss += loss_fn(logits, y_batch).item() * len(y_batch)
            correct += (logits.argmax(dim=1) == y_batch).sum().item()

    n_samples = len(loader.dataset)
    return total_loss / n_samples, correct / n_samples

In [ ]:
# 학습 전 상태에서 시작하도록 새 모델과 옵티마이저를 만듭니다
torch.manual_seed(0)  # 모델 초깃값과 미니배치 섞는 순서를 고정합니다
model = make_cnn()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 학습 전 첫 층 필터와 테스트 그림 앞 10장의 예측을 저장해 둡니다
filters_before = model[0].weight.detach().cpu().clone()
with torch.no_grad():
    predictions_before = model(X_test[:10].to(device)).argmax(dim=1).cpu()

# 에폭마다 학습하고, 학습 데이터 정확도를 기록합니다
history_rows = []
for epoch in range(1, 11):
    train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer)

    # 학습용 10,000장을 한 번에 넣어 예측하고 정확도를 구합니다
    with torch.no_grad():
        train_predictions = model(X_train.to(device)).argmax(dim=1).cpu()
    train_accuracy = (train_predictions == y_train).sum().item() / len(y_train)
    history_rows.append({"에폭": epoch, "학습 손실": train_loss, "학습 데이터 정확도": train_accuracy})

# 행 번호가 에폭이 되게 표를 만들고 소수 넷째 자리까지 봅니다
history_table = pd.DataFrame(history_rows).set_index("에폭")
display(history_table.round(4))

In [ ]:
# 에폭별 학습 손실과 학습 데이터 정확도
plt.figure(figsize=(14, 4.5))
plt.subplot(1, 2, 1)
plt.plot(history_table.index, history_table["학습 손실"], marker="o", linewidth=2)
plt.xticks(range(1, 11), fontsize=11)
plt.yticks(fontsize=11)
plt.xlabel("에폭", fontsize=12)
plt.ylabel("학습 손실", fontsize=12)
plt.title("CNN의 학습 손실", fontsize=14)
plt.subplot(1, 2, 2)
plt.plot(history_table.index, history_table["학습 데이터 정확도"], marker="o", linewidth=2, color="green")
plt.xticks(range(1, 11), fontsize=11)
plt.yticks(fontsize=11)
plt.xlabel("에폭", fontsize=12)
plt.ylabel("정확도", fontsize=12)
plt.title("CNN의 학습 데이터 정확도", fontsize=14)
plt.tight_layout()
plt.show()

**결과 읽기**
- **학습 손실:** 1에폭 0.7908에서 10에폭 0.0364로 줄었습니다. 첫 에폭에 가장 많이 줄고, 뒤로 갈수록 천천히 줄어듭니다.
- **학습 데이터 정확도:** 1에폭 뒤 0.9124에서 10에폭 뒤 0.9943으로 올랐습니다. 같은 설정의 지난 시간 DNN은 10에폭 뒤 0.9879였습니다.

### 5-2. 테스트 데이터로 한 번 확인하기

학습 데이터 정확도는 모델이 이미 본 그림의 결과이므로, 처음 보는 테스트 2,000장으로 한 번 평가합니다.

In [ ]:
# 학습을 마친 모델을 테스트 데이터로 한 번 평가합니다
test_loss, test_accuracy = evaluate(model, test_loader, loss_fn)
print("테스트 손실:", round(test_loss, 4), "| 테스트 정확도:", round(test_accuracy, 4))

**결과 읽기**
- **테스트 정확도 0.981:** 2,000장 가운데 38장만 틀렸습니다. 같은 그림, 같은 설정에서 지난 시간 DNN은 0.948(104장 틀림)이었습니다.
- **틀린 비율:** 학습 데이터는 약 0.6%(정확도 0.9943), 테스트 데이터는 약 1.9%입니다. 이 차이가 과적합인지는 교안 02에서 검증 데이터로 가립니다.

같은 테스트 그림 10장을 학습 전 모델과 학습 후 모델이 각각 어떻게 예측했는지 나란히 봅니다.

In [ ]:
# 위 줄은 학습 전 예측, 아래 줄은 학습 후 예측. 맞히면 초록, 틀리면 빨간 제목입니다
with torch.no_grad():
    predictions_after = model(X_test[:10].to(device)).argmax(dim=1).cpu()
prediction_rows = {"학습 전": predictions_before, "학습 후": predictions_after}

plt.figure(figsize=(14, 3.8))
for row, row_name in enumerate(prediction_rows):
    for k in range(10):
        plt.subplot(2, 10, row * 10 + k + 1)
        plt.imshow(X_test[k, 0], cmap="gray_r")

        # 맞혔는지에 따라 제목 색을 정합니다
        predicted = prediction_rows[row_name][k]
        if predicted == y_test[k]:
            title_color = "green"
        else:
            title_color = "red"
        plt.title(f"{row_name} {class_names[predicted]}", fontsize=12, color=title_color)
        plt.axis("off")
plt.tight_layout()
plt.show()

**그래프 읽기**
- **학습 전:** 무작위 파라미터라 10장을 모두 0으로 예측해, 정답이 0인 1장만 맞혔습니다.
- **학습 후:** 정답 1을 3으로 읽은 1장만 틀리고 9장을 맞혔습니다.

## 6. 필터, 특징 맵, 틀린 그림

정확도 숫자만으로는 모델이 어떤 특징으로 맞히는지 알 수 없으므로, 학습한 필터와 특징 맵, 틀린 그림을 그림으로 봅니다.

### 6-1. 첫 층의 필터 16개

첫 층 `model[0]`의 `weight`는 (16, 1, 3, 3), 학습으로 정해진 3 × 3 필터 16개입니다. 앞에서 손으로 정한 필터처럼, 빨간 칸(양수)에 획이 겹치고 파란 칸(음수)이 비어 있을 때 큰 값을 냅니다. 학습이 필터를 어떻게 바꿨는지 보려고, 학습 전에 저장한 필터와 같은 색 범위로 나란히 그립니다.

In [ ]:
# 학습한 첫 층 필터 16개(16, 1, 3, 3)를 CPU로 옮기고 기울기 기록을 뗍니다
filters_learned = model[0].weight.detach().cpu()
print("가장 큰 절댓값: 학습 전", round(filters_before.abs().max().item(), 4),
      "| 학습 후", round(filters_learned.abs().max().item(), 4))

# 두 줄이 같은 색 범위를 쓰고, 0이 흰색이 되도록 양쪽을 같게 둡니다
color_limit = max(filters_before.abs().max().item(), filters_learned.abs().max().item())

# 위 줄은 학습 전, 아래 줄은 학습 후 필터. 열 번호가 필터 번호입니다
filter_rows = {"학습 전": filters_before, "학습 후": filters_learned}
plt.figure(figsize=(14, 2.8))
for row, row_name in enumerate(filter_rows):
    for k in range(16):
        plt.subplot(2, 16, row * 16 + k + 1)
        plt.imshow(filter_rows[row_name][k, 0], cmap="bwr", vmin=-color_limit, vmax=color_limit)
        plt.xticks([])
        plt.yticks([])
        if row == 0:
            plt.title(f"{k}", fontsize=11)
        if k == 0:
            plt.ylabel(row_name, fontsize=11)
plt.suptitle("첫 층 필터 16개의 학습 전과 학습 후 (빨강: 양수, 파랑: 음수)", fontsize=14)
plt.tight_layout()
plt.show()

**그래프 읽기**
- **학습 후 색이 진해짐:** 가장 큰 절댓값이 0.3325에서 0.6288로 약 2배가 됐습니다.
- **무늬는 학습 전과 대체로 비슷함:** 학습은 무작위 초깃값에 있던 빨강·파랑 배치를 크게 바꾸지 않고 더 진하게 만들었습니다.
- **경계를 찾는 필터:** 한쪽이 빨갛고 반대쪽이 파란 필터는 그 방향의 경계(획이 시작하거나 끝나는 자리)에서 큰 값을 냅니다.
- **예:** 필터 11은 학습 전에도 위가 붉고 아래가 푸르스름했고, 학습 후 위 행 빨강, 아래 행 파랑이 뚜렷해져 가로 획의 아래쪽 경계를 찾습니다. 필터 14는 왼쪽 열이 파래져 획의 왼쪽 경계를 찾습니다.

### 6-2. 숫자 하나의 특징 맵

같은 필터 16개가 테스트 그림 한 장에서 무엇을 찾았는지, 첫 합성곱 묶음의 ReLU까지 통과한 특징 맵 16장을 봅니다.

- **`model[:2]`:** `nn.Sequential`도 리스트처럼 앞 두 층(합성곱, ReLU)만 잘라 쓸 수 있습니다.

In [ ]:
# 테스트 0번 그림 한 장을 첫 합성곱과 ReLU까지만 통과시킵니다. 모양 (1, 16, 28, 28)
sample = X_test[0:1].to(device)
with torch.no_grad():
    sample_maps = model[:2](sample).cpu()
print("정답:", y_test[0].item(), "| 특징 맵 모양:", tuple(sample_maps.shape))

# 첫 칸은 원본, 나머지 16칸은 필터마다의 특징 맵(진할수록 큰 값)
plt.figure(figsize=(14, 7.5))
plt.subplot(3, 6, 1)
plt.imshow(X_test[0, 0], cmap="gray_r")
plt.title("원본", fontsize=12)
plt.axis("off")
for k in range(16):
    plt.subplot(3, 6, k + 2)
    plt.imshow(sample_maps[0, k], cmap="gray_r")
    plt.title(f"필터 {k}", fontsize=12)
    plt.axis("off")
plt.tight_layout()
plt.show()

**그래프 읽기**
- **필터마다 다른 획:** 숫자 2 그림에서 필터 11의 맵은 가로 방향 획(아래 곡선, 위 획)만, 필터 14의 맵은 비스듬히 내려오는 획만 남깁니다. 위에서 본 학습 후 필터 모양과 같은 방향입니다.
- **획 전체가 진한 맵:** 필터 7, 15처럼 원본과 거의 같은 맵도 있습니다. 배경이 회색인 맵(필터 2, 9, 10, 12)은 절편이 양수라 빈 곳에서도 0보다 큰 값이 나온 것입니다.
- 둘째 합성곱 묶음의 필터는 이 16장을 한꺼번에 보므로, 여러 획을 묶은 더 큰 모양을 찾도록 학습됩니다.

### 6-3. 틀린 그림과 헷갈린 짝

어떤 그림을 틀렸고 어떤 숫자끼리 헷갈렸는지 보면 모델의 약점을 알 수 있으므로, 테스트 그림의 예측을 모아 틀린 그림과 혼동 행렬을 만듭니다.

- **혼동 행렬(confusion matrix):** 행은 정답, 열은 예측인 10 × 10 표입니다. 대각선(정답 = 예측)은 맞힌 수, 나머지 칸은 그 짝으로 틀린 수입니다. 예: 4행 9열이 5면 정답 4를 9로 예측한 그림이 5장입니다.

In [ ]:
# 테스트 그림 2,000장의 예측(로짓이 가장 큰 칸)을 구합니다
with torch.no_grad():
    test_predictions = model(X_test.to(device)).argmax(dim=1).cpu()

# 틀린 그림의 번호를 모읍니다
wrong_index = []
for i in range(len(y_test)):
    if test_predictions[i] != y_test[i]:
        wrong_index.append(i)
print("틀린 그림:", len(wrong_index), "/", len(y_test))

In [ ]:
# 틀린 그림 앞 10장을 정답과 예측으로 그립니다
plt.figure(figsize=(14, 2.6))
for k in range(10):
    i = wrong_index[k]
    plt.subplot(1, 10, k + 1)
    plt.imshow(X_test[i, 0], cmap="gray_r")
    plt.title(f"정답 {class_names[y_test[i]]}\n예측 {class_names[test_predictions[i]]}", fontsize=12)
    plt.axis("off")
plt.show()

In [ ]:
# 혼동 행렬: 행은 정답, 열은 예측. 그림 한 장마다 해당 칸에 1을 더합니다
confusion = torch.zeros(10, 10, dtype=torch.long)
for i in range(len(y_test)):
    confusion[y_test[i], test_predictions[i]] += 1
confusion_table = pd.DataFrame(confusion.tolist(), index=class_names, columns=class_names)
display(confusion_table)

# 대각선(맞힌 수)을 0으로 바꾼 뒤 가장 큰 칸이 가장 많이 헷갈린 짝입니다. 같은 값이면 모두 출력합니다
off_diagonal = confusion - torch.diag(torch.diag(confusion))
top_count = off_diagonal.max().item()
for true_label in range(10):
    for predicted_label in range(10):
        if off_diagonal[true_label, predicted_label] == top_count:
            print(f"가장 많이 헷갈린 짝: 정답 {class_names[true_label]}, 예측 {class_names[predicted_label]}인 그림 {top_count}장")

**결과 읽기**
- **틀린 그림 38장:** 앞 10장 가운데 5장이 정답 6이고, 2장은 0으로, 3장은 5로 예측했습니다. 획이 짧거나 고리가 열린 6, 옆에 점이 있는 1처럼 흔히 쓰는 모양과 다르게 쓴 그림이 많습니다.
- **혼동 행렬:** 대각선 밖은 대부분 0~2장입니다. 가장 많이 헷갈린 짝은 정답 6을 0으로, 5로 예측한 각 4장입니다.
- 특정 짝에 틀린 그림이 몰려 있으면, 그 숫자를 쓴 그림을 더 모으거나 그 두 숫자를 가르는 획을 모델이 보는지 특징 맵으로 확인하는 것이 다음 할 일입니다.

## 7. 모델 저장과 불러오기

학습한 모델을 서비스에 쓰거나 다음 날 이어서 쓰려면 매번 다시 학습할 수 없으므로, 학습한 파라미터를 파일로 저장했다가 불러옵니다.

- **`model.state_dict()`:** 파라미터 이름(`0.weight` 등)과 값(텐서)을 짝지은 딕셔너리입니다. 모델의 구조(어떤 층을 어떤 순서로 쌓았는지)는 들어 있지 않습니다.
- **`torch.save(딕셔너리, 경로)`:** 텐서가 든 딕셔너리를 파일로 저장합니다. 파일 이름 끝은 보통 `.pt`입니다.
- **`load_state_dict(딕셔너리)`:** 같은 구조로 새로 만든 모델에 저장한 값을 넣습니다. 그래서 불러올 때도 `make_cnn()`으로 같은 구조를 먼저 만듭니다.
- **`torch.load(경로, map_location=device)`:** 파일의 텐서를 현재 계산 장치로 옮겨 읽습니다. GPU에서 저장한 파일을 CPU에서 읽을 때도 이 인자로 읽습니다.

In [ ]:
# state_dict의 키(파라미터 이름)와 텐서 모양. 이 모델에는 층마다 weight와 bias만 들어 있습니다
state = model.state_dict()
for key in state:
    print(key, tuple(state[key].shape))

In [ ]:
# output 폴더는 저장소에 올리지 않는 폴더라, 처음 실행할 때 만듭니다
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)

# 학습한 파라미터를 파일로 저장합니다
model_path = output_dir / "mnist_cnn.pt"
torch.save(model.state_dict(), model_path)
print("저장한 파일:", model_path.name, "|", round(model_path.stat().st_size / 1024, 1), "KB")

In [ ]:
# 같은 구조의 새 모델: 파라미터가 무작위라 아직 숫자를 구별하지 못합니다
loaded_model = make_cnn()
_, before_accuracy = evaluate(loaded_model, test_loader, loss_fn)
print("불러오기 전 테스트 정확도:", round(before_accuracy, 4))

# 저장한 파일을 읽어 새 모델에 넣습니다
loaded_model.load_state_dict(torch.load(model_path, map_location=device))
_, after_accuracy = evaluate(loaded_model, test_loader, loss_fn)
print("불러온 뒤 테스트 정확도:", round(after_accuracy, 4))

# 불러온 모델의 예측이 학습한 모델의 예측과 2,000장 모두 같은지 확인합니다
with torch.no_grad():
    loaded_predictions = loaded_model(X_test.to(device)).argmax(dim=1).cpu()
print("예측이 모두 같은가:", torch.equal(loaded_predictions, test_predictions))

**결과 읽기**
- **불러오기 전:** 무작위 파라미터라 정확도가 무작위로 고를 때의 정확도(약 0.1)와 같은 수준입니다.
- **불러온 뒤:** 학습을 마친 모델과 같은 0.981이고, 2,000장의 예측이 모두 같습니다. 파일 크기는 83.0KB입니다. 파라미터 20,490개 × 4바이트(float32) = 약 80KB에 파라미터 이름 같은 정보가 더해진 크기입니다.
- **실무 판단:** 파일에는 구조가 없으므로, 모델을 만드는 코드(`make_cnn`)도 함께 보관해야 다른 컴퓨터에서 불러올 수 있습니다.

## 8. 🖐️ 함께 따라하기: 옷 사진 분류

본실습의 `make_cnn()`과 학습 함수를 옷 사진에 적용해, 같은 CNN이 손글씨가 아닌 그림에서도 작동하는지와 어떤 옷끼리 헷갈리는지를 직접 작성해 확인합니다.

**본실습과 다른 점**
- **데이터:** 옷 사진 [Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist)(Xiao, Rasul & Vollgraf, 2017, MIT 라이선스)에서 고른 학습용 10,000장과 테스트용 2,000장입니다.
- **입력 크기:** 28 × 28 흑백, 종류 10개로 MNIST와 같아 `make_cnn()`을 그대로 씁니다.
- **읽는 법:** 숫자보다 겉모양이 비슷한 종류가 많아, 혼동 행렬에서 어떤 옷끼리 헷갈리는지 봅니다.

**실습 진행 방법:** 작성 셀의 `...`만 주석에 맞춰 채웁니다. 완성 코드는 `정답/교안_01_정답.ipynb`의 같은 절에 있습니다.

**설정**
- 배치 크기 64(학습용만 섞기), Adam 학습률 0.001, 10에폭, 모델을 만들기 직전 `torch.manual_seed(0)`

### 8-1. 데이터 읽기

옷 사진 10종류를 한 장씩 봅니다.

In [ ]:
# 옷 사진: 그림과 정답 텐서 4개, 종류 이름 10개
fashion = torch.load(data_dir / "따라하기" / "fashion_small.pt")
fashion_names = fashion["class_names"]
print("학습용:", fashion["train_images"].shape, "| 테스트용:", fashion["test_images"].shape)
print("종류:", fashion_names)

# 종류마다 학습용 첫 그림 한 장씩 그립니다
plt.figure(figsize=(14, 2.6))
for label in range(10):
    # 그 종류가 처음 나오는 학습용 그림 번호를 앞에서부터 찾습니다
    first_index = 0
    while fashion["train_labels"][first_index] != label:
        first_index += 1
    plt.subplot(1, 10, label + 1)
    plt.imshow(fashion["train_images"][first_index], cmap="gray_r")
    plt.title(fashion_names[label], fontsize=12)
    plt.axis("off")
plt.show()

### 8-2. CNN 입력 모양과 DataLoader

In [ ]:
# 🖐️ 함께 따라하기: 그림은 0~1 실수로 바꾸고 채널 축을 끼워 (N, 1, 28, 28)로, 정답은 그대로 꺼내세요
X_fashion_train = ...
X_fashion_test = ...
y_fashion_train = ...
y_fashion_test = ...
print("X_fashion_train:", X_fashion_train.shape, X_fashion_train.dtype, "| y_fashion_train:", y_fashion_train.shape)

# 🖐️ 학습용과 테스트용 DataLoader를 만드세요(배치 크기 64, 학습용만 섞습니다)
fashion_train_loader = ...
fashion_test_loader = ...
print("학습 미니배치 수:", len(fashion_train_loader), "| 테스트 미니배치 수:", len(fashion_test_loader))

### 8-3. CNN 만들고 학습하기

본실습의 `make_cnn()`, `loss_fn`, `train_one_epoch`, `evaluate`를 그대로 씁니다.

In [ ]:
# 🖐️ 함께 따라하기: 본실습과 같은 CNN과 Adam 옵티마이저(학습률 0.001)를 만드세요
torch.manual_seed(0)  # 모델 초깃값과 미니배치 섞는 순서를 고정합니다
fashion_model = ...
fashion_optimizer = ...

# 🖐️ 10에폭 학습하며 에폭마다 학습 손실과 학습 데이터 정확도를 기록하세요
fashion_rows = []
for epoch in range(1, 11):
    train_loss = ...

    # 🖐️ 학습용 사진 전체를 한 번에 넣어 예측(argmax)하세요
    with torch.no_grad():
        train_predictions = ...
    train_accuracy = (train_predictions == y_fashion_train).sum().item() / len(y_fashion_train)
    fashion_rows.append({"에폭": epoch, "학습 손실": train_loss, "학습 데이터 정확도": train_accuracy})

# 행 번호가 에폭이 되게 표를 만들고 소수 넷째 자리까지 봅니다
fashion_history = pd.DataFrame(fashion_rows).set_index("에폭")
display(fashion_history.round(4))

In [ ]:
# 에폭별 학습 손실과 학습 데이터 정확도
plt.figure(figsize=(14, 4.5))
plt.subplot(1, 2, 1)
plt.plot(fashion_history.index, fashion_history["학습 손실"], marker="o", linewidth=2)
plt.xticks(range(1, 11), fontsize=11)
plt.yticks(fontsize=11)
plt.xlabel("에폭", fontsize=12)
plt.ylabel("학습 손실", fontsize=12)
plt.title("옷 사진 CNN의 학습 손실", fontsize=14)
plt.subplot(1, 2, 2)
plt.plot(fashion_history.index, fashion_history["학습 데이터 정확도"], marker="o", linewidth=2, color="green")
plt.xticks(range(1, 11), fontsize=11)
plt.yticks(fontsize=11)
plt.xlabel("에폭", fontsize=12)
plt.ylabel("정확도", fontsize=12)
plt.title("옷 사진 CNN의 학습 데이터 정확도", fontsize=14)
plt.tight_layout()
plt.show()

### 8-4. 테스트 정확도와 혼동 행렬

학습을 마친 모델을 테스트 데이터로 한 번 평가하고, 본실습 6-3처럼 혼동 행렬로 헷갈린 짝을 봅니다.

In [ ]:
# 🖐️ 함께 따라하기: 테스트 데이터로 한 번 평가하고, 테스트 사진 2,000장의 예측(argmax)을 구하세요
fashion_test_loss, fashion_test_accuracy = ...
with torch.no_grad():
    fashion_predictions = ...
print("테스트 정확도:", round(fashion_test_accuracy, 4))

# 🖐️ 혼동 행렬을 만드세요(행은 정답, 열은 예측). 0으로 채운 10 x 10 표에서 시작합니다
fashion_confusion = ...
for i in range(len(y_fashion_test)):
    ...
fashion_table = pd.DataFrame(fashion_confusion.tolist(), index=fashion_names, columns=fashion_names)
display(fashion_table)

# 사진 2,000장이 한 칸씩 들어갔다면 표의 합은 2000입니다
print("표의 합:", fashion_confusion.sum().item())

In [ ]:
# 종류마다 맞힌 비율 = 대각선 칸 / 그 행(정답)의 합
class_rows = []
for label in range(10):
    class_rows.append({"종류": fashion_names[label],
                       "맞힌 비율": fashion_confusion[label, label].item() / fashion_confusion[label].sum().item()})
class_table = pd.DataFrame(class_rows)

plt.figure(figsize=(14, 4))
plt.bar(class_table["종류"], class_table["맞힌 비율"], color="steelblue")
for k in range(10):
    ratio = class_table["맞힌 비율"][k]
    plt.text(k, ratio + 0.02, f"{ratio:.2f}", ha="center", fontsize=11)  # 막대 위에 비율을 적습니다
plt.xticks(fontsize=12)
plt.yticks(fontsize=11)
plt.ylim(0, 1.1)
plt.ylabel("맞힌 비율", fontsize=12)
plt.title("옷 종류마다 맞힌 비율(테스트)", fontsize=14)
plt.show()

In [ ]:
# 정답이 셔츠인데 다른 옷으로 틀린 사진 10장을 그립니다
shirt = fashion_names.index("셔츠")
shirt_wrong = []
for i in range(len(y_fashion_test)):
    if y_fashion_test[i] == shirt and fashion_predictions[i] != shirt:
        shirt_wrong.append(i)

plt.figure(figsize=(14, 2.6))
for k in range(10):
    i = shirt_wrong[k]
    plt.subplot(1, 10, k + 1)
    plt.imshow(X_fashion_test[i, 0], cmap="gray_r")
    plt.title(f"정답 셔츠\n예측 {fashion_names[fashion_predictions[i]]}", fontsize=12)
    plt.axis("off")
plt.show()

**확인 퀴즈**: 옷 사진 CNN에서 맞힌 비율이 가장 낮은 종류는 무엇이고, 무엇과 헷갈릴까요? 맞힌 비율 막대와 틀린 셔츠 사진으로 설명해 봅니다.

<details><summary>정답 보기</summary>

테스트 정확도는 약 0.87로 손글씨(0.981)보다 낮습니다.

- **약한 종류:** 스웨터, 코트, 셔츠가 약 0.73~0.74로 가장 낮습니다.
- **헷갈린 짝:** 이 셋과 티셔츠 사이에 틀린 그림이 16~31장씩 몰려 있습니다(코트를 셔츠로, 스웨터를 셔츠로, 셔츠를 티셔츠로). 윗옷이 아닌 짝으로는 앵클부츠를 운동화로 14장 틀렸습니다.
- **이유:** 28 × 28 흑백 사진에서는 깃, 단추, 소매 길이 같은 작은 차이가 몇 픽셀뿐이라, 겉모양이 비슷한 윗옷끼리 구별하기 어렵습니다. 바지·운동화는 0.96 넘게 맞힙니다.
- **실무 판단:** 헷갈리는 종류의 사진을 더 모으거나, 해상도가 높은 사진을 쓰는 것이 다음 할 일입니다. 혼동 행렬에서 큰 칸을 먼저 확인합니다.

</details>

## 9. 핵심 정리

- **CNN 입력 모양:** (그림 수, 채널, 높이, 너비)입니다. 흑백은 채널 1, 컬러는 3이며, 흑백 그림 묶음은 `unsqueeze(1)`로 채널 축을 끼웁니다.
- **합성곱:** 3 × 3 필터를 한 칸씩 옮기며 겹친 픽셀과 곱해서 더합니다. 필터 숫자가 찾는 모양을 정하고(세로선 필터는 세로줄에 6, 가로줄에 0), CNN은 이 숫자를 학습합니다. `nn.Conv2d(입력 채널, 출력 채널, kernel_size=3, padding=1)`입니다.
- **출력 크기와 채널:** 입력 크기 $n$, 필터 크기 $f$, padding $p$, stride 1일 때 출력 크기는 $n - f + 1 + 2p$이고, `padding=1`이면 크기가 그대로입니다. 출력 채널 수 = 필터 수 = 특징 맵 수입니다.
- **최댓값 풀링:** 2 × 2 구역마다 가장 큰 값을 남겨 가로·세로를 절반으로 줄입니다(`nn.MaxPool2d(2)`, stride 2, 28 → 14 → 7).
- **CNN 쌓기:** [합성곱 → ReLU → 풀링]을 두 번 쌓고 `nn.Flatten()` → `nn.Linear(32 * 7 * 7, 10)`으로 분류합니다. 층마다 모양을 출력해 `nn.Linear`의 입력 크기를 맞춥니다. 마지막 층이 내는 값은 로짓(logit) 10개입니다. 이 교안의 CNN은 파라미터 20,490개로 손글씨 테스트 정확도 0.981이었습니다(지난 시간 DNN은 109,386개, 0.948).
- **학습 결과 읽기:** 학습한 필터, 특징 맵, 틀린 그림, 혼동 행렬(confusion matrix)로 모델이 어떤 특징을 쓰고 어떤 숫자끼리 헷갈리는지 확인합니다. 옷 사진에서는 스웨터, 코트, 셔츠의 맞힌 비율이 약 0.73~0.74로 낮았습니다.
- **저장과 불러오기:** `torch.save(model.state_dict(), 경로)`로 파라미터만 저장하고, 같은 구조로 만든 모델에 `load_state_dict(torch.load(경로, map_location=device))`로 넣습니다.

교안 02에서는 학습 데이터를 줄였을 때 생기는 과적합을 검증 데이터로 찾고, Dropout·weight decay·데이터 증강·조기 종료로 줄인 뒤 같은 학습을 PyTorch Lightning으로 정리합니다.